# Libs

In [13]:
from google.colab import drive
from PIL import Image
from math import floor
import pandas as pd
import os
drive.mount('/content/drive')
BASE_DIR = "/content/drive/Shareddrives/CV_Project"
WORK_FOLDER = "diffusion"
DATA_DIR = os.path.join(BASE_DIR, WORK_FOLDER, "data")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Unzip Downloaded Zip

In [ ]:
!mkdir "{DATA_DIR}"

mkdir: cannot create directory ‘/content/drive/Shareddrives/CV_Project/diffusion/data’: File exists


In [ ]:
!unzip "/content/drive/Shareddrives/CV_Project/archive.zip" -d "{DATA_DIR}"

Archive:  /content/drive/Shareddrives/CV_Project/archive.zip
  inflating: /content/drive/Shareddrives/CV_Project/diffusion/data/IndustryBiscuit/Annotations.csv  
  inflating: /content/drive/Shareddrives/CV_Project/diffusion/data/IndustryBiscuit/DatasetFolder.py  
  inflating: /content/drive/Shareddrives/CV_Project/diffusion/data/IndustryBiscuit/Images/0001.jpg  
  inflating: /content/drive/Shareddrives/CV_Project/diffusion/data/IndustryBiscuit/Images/0002.jpg  
  inflating: /content/drive/Shareddrives/CV_Project/diffusion/data/IndustryBiscuit/Images/0003.jpg  
  inflating: /content/drive/Shareddrives/CV_Project/diffusion/data/IndustryBiscuit/Images/0004.jpg  
  inflating: /content/drive/Shareddrives/CV_Project/diffusion/data/IndustryBiscuit/Images/0005.jpg  
  inflating: /content/drive/Shareddrives/CV_Project/diffusion/data/IndustryBiscuit/Images/0006.jpg  
  inflating: /content/drive/Shareddrives/CV_Project/diffusion/data/IndustryBiscuit/Images/0007.jpg  
  inflating: /content/drive/S

# Organize Train, Val, Test

In [ ]:
!ls -l "{DATA_DIR}/IndustryBiscuit"

total 123
-rw------- 1 root root 112576 Oct 11  2022 Annotations.csv
-rw------- 1 root root   4792 Oct 11  2022 DatasetFolder.py
drwx------ 5 root root   4096 Nov 30 03:51 DS
drwx------ 2 root root   4096 Nov 30 03:50 Images


In [ ]:
# This script prepares the dataset according to the set parameters and divides it into the required directory structure

# Specify the image, annotations and destination path
imPath = os.path.join(DATA_DIR, 'IndustryBiscuit/Images')
anPath = os.path.join(DATA_DIR, 'IndustryBiscuit/Annotations.csv')
dsPath = os.path.join(DATA_DIR, 'IndustryBiscuit/DS')
print(dsPath)

# Set the dataset parameters
nTrain_ok = 1000
nTrain_nok = 0
nValid_ok = 100
nValid_nok = 100
nTest_ok = 100
nTest_nok = 100

# Defect ratios
rNComplete = 0.4
rSObject = 0.3
rCDefect = 0.3

# Counters initialization
cTrain_ok = 0
cValid_ok = 0
cTest_ok = 0

cTrainNC_nok = 0
cValidNC_nok = 0
cTestNC_nok = 0

cTrainSO_nok = 0
cValidSO_nok = 0
cTestSO_nok = 0

cTrainCD_nok = 0
cValidCD_nok = 0
cTestCD_nok = 0

# Defect limits
nTrNC = floor(nTrain_nok * rNComplete)
nVaNC = floor(nValid_nok * rNComplete)
nTeNC = floor(nTest_nok * rNComplete)

nTrSO = floor(nTrain_nok * rSObject)
nVaSO = floor(nValid_nok * rSObject)
nTeSO = floor(nTest_nok * rSObject)

nTrCD = floor(nTrain_nok * rCDefect)
nVaCD = floor(nValid_nok * rCDefect)
nTeCD = floor(nTest_nok * rCDefect)

# Create the directories for the image storage
if not os.path.exists(dsPath):
    print("Creating folder structure...")

    # Create the folder structure
    os.mkdir(dsPath)
    os.mkdir(dsPath + '/train')
    os.mkdir(dsPath + '/train' + '/ok')
    os.mkdir(dsPath + '/train' + '/nok')
    os.mkdir(dsPath + '/valid')
    os.mkdir(dsPath + '/valid' + '/ok')
    os.mkdir(dsPath + '/valid' + '/nok')
    os.mkdir(dsPath + '/test')
    os.mkdir(dsPath + '/test' + '/ok')
    os.mkdir(dsPath + '/test' + '/nok')

    # Load the filenames and the annotation from the .csv file
    data = pd.read_csv(anPath, usecols= ['file', 'classDescription'])

    augm = 1226

    for key in range(1, 1226):
        print(key)

        for temp in range(0, 4):

            if temp == 0:
                index = key
            else:
                index = augm
                augm += 1

            value = data.iloc[index - 1, :]

            # Open the image file
            im = Image.open(os.path.join(imPath, value[0]))

            # Split the images to the categories
            if value[1] == "Defect_No":
                if (cTrain_ok < nTrain_ok):
                    im.save(os.path.join(dsPath + '/train' + '/ok', value[0]), format='jpeg')
                    cTrain_ok += 1
                elif (cValid_ok < nValid_ok):
                    im.save(os.path.join(dsPath + '/valid' + '/ok', value[0]), format='jpeg')
                    cValid_ok += 1
                elif (cTest_ok < nTest_ok):
                    im.save(os.path.join(dsPath + '/test' + '/ok', value[0]), format='jpeg')
                    cTest_ok += 1

            elif value[1] == "Defect_Shape":
                if (cTrainNC_nok < nTrNC):
                    im.save(os.path.join(dsPath + '/train' + '/nok', value[0]), format='jpeg')
                    cTrainNC_nok += 1
                elif (cValidNC_nok < nVaNC):
                    im.save(os.path.join(dsPath + '/valid' + '/nok', value[0]), format='jpeg')
                    cValidNC_nok += 1
                elif (cTestNC_nok < nTeNC):
                    im.save(os.path.join(dsPath + '/test' + '/nok', value[0]), format='jpeg')
                    cTestNC_nok += 1

            elif value[1] == "Defect_Object":
                if (cTrainSO_nok < nTrSO):
                    im.save(os.path.join(dsPath + '/train' + '/nok', value[0]), format='jpeg')
                    cTrainSO_nok += 1
                elif (cValidSO_nok < nVaSO):
                    im.save(os.path.join(dsPath + '/valid' + '/nok', value[0]), format='jpeg')
                    cValidSO_nok += 1
                elif (cTestSO_nok < nTeSO):
                    im.save(os.path.join(dsPath + '/test' + '/nok', value[0]), format='jpeg')
                    cTestSO_nok += 1

            elif value[1] == "Defect_Color":
                if (cTrainCD_nok < nTrCD):
                    im.save(os.path.join(dsPath + '/train' + '/nok', value[0]), format='jpeg')
                    cTrainCD_nok += 1
                elif (cValidCD_nok < nVaCD):
                    im.save(os.path.join(dsPath + '/valid' + '/nok', value[0]), format='jpeg')
                    cValidCD_nok += 1
                elif (cTestCD_nok < nTeCD):
                    im.save(os.path.join(dsPath + '/test' + '/nok', value[0]), format='jpeg')
                    cTestCD_nok += 1

    # Print dataset statistics TODO

    print ("Dataset created successfully...")

else:
    print("Folder structure with the dataset already exists...")

/content/drive/Shareddrives/CV_Project/diffusion/data/IndustryBiscuit/DS
Folder structure with the dataset already exists...
